# CartFlow (P06) — Week 10: Controlled Streaming Simulation

**Notebook:** `notebooks/07_streaming_simulation.ipynb`

This notebook implements the approved Week-10 streaming contract in Databricks:

- process `order_status_drop_01.json`, then `order_status_drop_02.json` incrementally;
- use an explicit event schema;
- use the required stable Auto Loader checkpoint;
- preserve physical source filename + physical line number + line identity;
- apply a 30-minute event-time watermark;
- apply event-ID and order/status/sequence deduplication;
- route malformed/schema-error, late, duplicate, orphaned and invalid-transition events;
- retain trusted and quarantined outcomes;
- prove that a no-new-file rerun with the same checkpoints adds zero new output.

**Run order:** execute cells from top to bottom. **Do not use Run All.**

**Week-10 boundary:** this notebook does not rebuild Gold or modify the Week-9 Power BI dashboard.

## Contract and implementation decisions

The approved Week-10 playbook requires trusted/quarantined outcomes to retain **physical filename + line identity**. Because the source files are JSON Lines, this notebook uses Auto Loader with the `text` source so every physical JSON line is retained as `_raw_json`. A small static manifest assigns the exact physical line number from each approved 100-line file. The streaming path itself remains Auto Loader / Structured Streaming.

The event payload is then parsed with an **explicit 14-field event schema**. This allows malformed JSON, missing/invalid typed fields, and unexpected JSON keys to be routed instead of silently disappearing.

The 30-minute watermark is attached to the downstream Structured Streaming DataFrame. Late-event routing is made explicit inside `foreachBatch` so late rows are retained in quarantine rather than silently dropped. The controlled files are processed one file per microbatch (`maxFilesPerTrigger=1`), which makes the physical file-line order available for deterministic within-file late detection.

A future-dated event is treated as an additional `future_event_time` quality reason so it cannot incorrectly advance the watermark state and make an entire later drop appear late. This is an additional defensive check; the required Week-10 categories remain covered.

## 0. Configuration — Databricks only

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from datetime import datetime, timezone

BASE_DIR = "/Volumes/p06/default/cartflow/week10"
INPUT_DIR = f"{BASE_DIR}/input"
HOLD_DIR = f"{BASE_DIR}/input_hold"

DROP01_PATH = f"{INPUT_DIR}/order_status_drop_01.json"
DROP02_PATH = f"{INPUT_DIR}/order_status_drop_02.json"
DROP02_HOLD_PATH = f"{HOLD_DIR}/order_status_drop_02.json"

# Required stable Week-10 checkpoint.
BRONZE_CHECKPOINT_DIR = f"{BASE_DIR}/stable/checkpoints/cartflow_order_status_v1"

# Separate schema and downstream-quality checkpoint locations.
SCHEMA_LOCATION = f"{BASE_DIR}/stable/schema/cartflow_order_status_v1"
QUALITY_CHECKPOINT_DIR = f"{BASE_DIR}/stable/checkpoints/cartflow_order_status_quality_v1"

BRONZE_TABLE = "p06.default.cartflow_week10_bronze_order_status"
TRUSTED_TABLE = "p06.default.cartflow_week10_trusted_order_status"
QUARANTINE_TABLE = "p06.default.cartflow_week10_quarantine_order_status"

EVENT_LEDGER_TABLE = "p06.default.cartflow_week10_event_dedup_ledger"
BUSINESS_LEDGER_TABLE = "p06.default.cartflow_week10_business_dedup_ledger"

CONTROL_TABLE = "p06.default.cartflow_week10_stream_control"
AUDIT_TABLE = "p06.default.cartflow_week10_run_audit"

WATERMARK_DELAY = "30 minutes"
MAX_FILES_PER_TRIGGER = 1

# Safety switch. Keep False for a normal run.
# Set True only when you intentionally want a completely fresh Week-10 state.
RESET_STATE = False

for path in [
    HOLD_DIR,
    f"{BASE_DIR}/stable",
    SCHEMA_LOCATION,
    BRONZE_CHECKPOINT_DIR,
    QUALITY_CHECKPOINT_DIR,
]:
    dbutils.fs.mkdirs(path)

print("INPUT_DIR:", INPUT_DIR)
print("DROP01_PATH:", DROP01_PATH)
print("DROP02_PATH:", DROP02_PATH)
print("BRONZE_CHECKPOINT_DIR:", BRONZE_CHECKPOINT_DIR)
print("QUALITY_CHECKPOINT_DIR:", QUALITY_CHECKPOINT_DIR)

## 1. Optional clean start and state guard

This cell is intentionally safe by default. If an earlier Week-10 attempt created tables/checkpoints with a different schema, the notebook stops instead of mixing old state with the corrected implementation.

In [ ]:
def path_has_entries(path):
    try:
        return len(dbutils.fs.ls(path)) > 0
    except Exception:
        return False

week10_tables = [
    BRONZE_TABLE,
    TRUSTED_TABLE,
    QUARANTINE_TABLE,
    EVENT_LEDGER_TABLE,
    BUSINESS_LEDGER_TABLE,
    CONTROL_TABLE,
    AUDIT_TABLE,
]

if RESET_STATE:
    for table_name in week10_tables:
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")

    for path in [
        BRONZE_CHECKPOINT_DIR,
        QUALITY_CHECKPOINT_DIR,
        SCHEMA_LOCATION,
    ]:
        dbutils.fs.rm(path, True)
        dbutils.fs.mkdirs(path)

    print("RESET_STATE=True: Week-10 Delta tables and streaming state were cleared.")
else:
    existing_tables = [
        table_name
        for table_name in week10_tables
        if spark.catalog.tableExists(table_name)
    ]

    existing_state_paths = [
        path
        for path in [BRONZE_CHECKPOINT_DIR, QUALITY_CHECKPOINT_DIR, SCHEMA_LOCATION]
        if path_has_entries(path)
    ]

    if existing_tables or existing_state_paths:
        raise RuntimeError(
            "Existing Week-10 state was detected. "
            "For this corrected notebook, use a clean run: set RESET_STATE=True "
            "in the configuration cell, run this cell once, then continue. "
            f"Existing tables={existing_tables}; existing state paths={existing_state_paths}"
        )

    print("RESET_STATE=False: no prior Week-10 tables or streaming state detected.")

## 2. Prepare the controlled two-drop sequence

Drop 02 is held outside the landing directory during the Drop-01 run. Drop 01 is never rewritten. After Drop-01 evidence is captured, Drop 02 is released into the same landing directory.

In [ ]:
input_paths = {x.path for x in dbutils.fs.ls(INPUT_DIR)}
hold_paths = {x.path for x in dbutils.fs.ls(HOLD_DIR)}

if DROP01_PATH not in input_paths:
    raise FileNotFoundError(
        "Drop 01 is missing from the Week-10 landing directory: " + DROP01_PATH
    )

if DROP02_PATH in input_paths:
    dbutils.fs.mv(DROP02_PATH, DROP02_HOLD_PATH)
    print("Drop 02 moved to the hold directory. Drop 01 remains untouched in input.")
elif DROP02_HOLD_PATH in hold_paths:
    print("Drop 02 is already in the hold directory. No move required.")
else:
    raise FileNotFoundError(
        "Drop 02 was not found in either input or hold: " + DROP02_PATH
    )

print("Input directory:")
display(dbutils.fs.ls(INPUT_DIR))

print("Hold directory:")
display(dbutils.fs.ls(HOLD_DIR))

## 3. Build the physical-line manifest

The playbook identifies each status-drop source as 100 physical JSON Lines. The manifest is only an audit/lineage helper; it does not rewrite the source files.

In [ ]:
manifest_schema = T.StructType([
    T.StructField("source_file_name", T.StringType(), False),
    T.StructField("source_line_no", T.LongType(), False),
    T.StructField("raw_json", T.StringType(), False),
    T.StructField("source_line_identity", T.StringType(), False),
])

manifest_rows = []

def add_manifest_rows(path):
    file_name = path.rsplit("/", 1)[-1]
    raw_text = dbutils.fs.head(path, 2 * 1024 * 1024)
    lines = raw_text.splitlines()

    if len(lines) != 100:
        raise RuntimeError(
            f"{file_name} contains {len(lines)} physical lines; "
            "the approved Week-10 source contract expects 100."
        )

    if len(lines) != len(set(lines)):
        raise RuntimeError(
            f"{file_name} contains identical raw JSON lines, so a simple "
            "file+line manifest would not be one-to-one."
        )

    for line_no, raw_json in enumerate(lines, start=1):
        manifest_rows.append((
            file_name,
            line_no,
            raw_json,
            f"{file_name}:{line_no}",
        ))

add_manifest_rows(DROP01_PATH)
add_manifest_rows(DROP02_HOLD_PATH)

line_manifest = spark.createDataFrame(manifest_rows, manifest_schema).cache()

manifest_duplicate_count = (
    line_manifest
    .groupBy("source_file_name", "source_line_no")
    .count()
    .where("count > 1")
    .count()
)

if manifest_duplicate_count != 0:
    raise RuntimeError("Physical-line manifest contains duplicate file+line identities.")

display(
    line_manifest
    .groupBy("source_file_name")
    .agg(
        F.count("*").alias("physical_lines"),
        F.min("source_line_no").alias("first_line"),
        F.max("source_line_no").alias("last_line"),
    )
    .orderBy("source_file_name")
)

## 4. Explicit event schema

The event payload schema is explicit and is never inferred.

In [ ]:
event_schema = T.StructType([
    T.StructField("event_id", T.StringType(), True),
    T.StructField("schema_version", T.IntegerType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
    T.StructField("event_type", T.StringType(), True),
    T.StructField("order_id", T.StringType(), True),
    T.StructField("seller_id", T.StringType(), True),
    T.StructField("order_item_id", T.StringType(), True),
    T.StructField("previous_status", T.StringType(), True),
    T.StructField("new_status", T.StringType(), True),
    T.StructField("status_reason", T.StringType(), True),
    T.StructField("promised_delivery_ts", T.TimestampType(), True),
    T.StructField("location_code", T.StringType(), True),
    T.StructField("event_sequence_no", T.IntegerType(), True),
    T.StructField("producer_run_id", T.StringType(), True),
])

raw_line_schema = T.StructType([
    T.StructField("value", T.StringType(), True),
])

raw_map_schema = T.MapType(T.StringType(), T.StringType(), True)

ALLOWED_EVENT_FIELDS = [
    "event_id",
    "schema_version",
    "event_ts",
    "event_type",
    "order_id",
    "seller_id",
    "order_item_id",
    "previous_status",
    "new_status",
    "status_reason",
    "promised_delivery_ts",
    "location_code",
    "event_sequence_no",
    "producer_run_id",
]

REQUIRED_JSON_KEYS = ALLOWED_EVENT_FIELDS

print("Explicit event fields:", ALLOWED_EVENT_FIELDS)

## 5. Verify the approved Trusted Silver reference keys

In [ ]:
required_refs = {
    "p06.default.trusted_silver_orders": "order_id",
    "p06.default.trusted_silver_order_items": "order_item_id",
    "p06.default.trusted_silver_sellers": "seller_id",
}

for table_name, key_col in required_refs.items():
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Required Trusted Silver table is missing: {table_name}")

    columns = {field.name for field in spark.table(table_name).schema.fields}
    if key_col not in columns:
        raise RuntimeError(
            f"Required reference key {key_col} is missing from {table_name}"
        )

orders_ref = (
    spark.table("p06.default.trusted_silver_orders")
    .select("order_id")
    .where(F.col("order_id").isNotNull())
    .dropDuplicates()
)

items_ref = (
    spark.table("p06.default.trusted_silver_order_items")
    .select("order_item_id")
    .where(F.col("order_item_id").isNotNull())
    .dropDuplicates()
)

sellers_ref = (
    spark.table("p06.default.trusted_silver_sellers")
    .select("seller_id")
    .where(F.col("seller_id").isNotNull())
    .dropDuplicates()
)

print("Trusted Silver reference tables and keys are available.")
display(
    spark.createDataFrame(
        [
            ("trusted_silver_orders", orders_ref.count()),
            ("trusted_silver_order_items", items_ref.count()),
            ("trusted_silver_sellers", sellers_ref.count()),
        ],
        ["reference_table", "distinct_reference_keys"],
    )
)

## 6. Create the Week-10 Delta tables

The output tables include physical filename, physical line number, and a stable file+line identity.

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    event_id STRING,
    schema_version INT,
    event_ts TIMESTAMP,
    event_type STRING,
    order_id STRING,
    seller_id STRING,
    order_item_id STRING,
    previous_status STRING,
    new_status STRING,
    status_reason STRING,
    promised_delivery_ts TIMESTAMP,
    location_code STRING,
    event_sequence_no INT,
    producer_run_id STRING,
    raw_json STRING,
    source_file_path STRING,
    source_file_name STRING,
    source_line_no BIGINT,
    source_line_identity STRING,
    schema_error_flag BOOLEAN,
    schema_error_detail STRING,
    ingested_at TIMESTAMP,
    bronze_batch_id BIGINT
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TRUSTED_TABLE} (
    event_id STRING,
    schema_version INT,
    event_ts TIMESTAMP,
    event_type STRING,
    order_id STRING,
    seller_id STRING,
    order_item_id STRING,
    previous_status STRING,
    new_status STRING,
    status_reason STRING,
    promised_delivery_ts TIMESTAMP,
    location_code STRING,
    event_sequence_no INT,
    producer_run_id STRING,
    raw_json STRING,
    source_file_path STRING,
    source_file_name STRING,
    source_line_no BIGINT,
    source_line_identity STRING,
    quality_reason STRING,
    trusted_at TIMESTAMP,
    quality_batch_id BIGINT
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} (
    event_id STRING,
    schema_version INT,
    event_ts TIMESTAMP,
    event_type STRING,
    order_id STRING,
    seller_id STRING,
    order_item_id STRING,
    previous_status STRING,
    new_status STRING,
    status_reason STRING,
    promised_delivery_ts TIMESTAMP,
    location_code STRING,
    event_sequence_no INT,
    producer_run_id STRING,
    raw_json STRING,
    source_file_path STRING,
    source_file_name STRING,
    source_line_no BIGINT,
    source_line_identity STRING,
    quality_reason STRING,
    quality_detail STRING,
    quarantined_at TIMESTAMP,
    quality_batch_id BIGINT
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {EVENT_LEDGER_TABLE} (
    event_id STRING,
    first_source_file_name STRING,
    first_source_line_no BIGINT,
    first_source_line_identity STRING,
    first_outcome_reason STRING,
    first_seen_event_ts TIMESTAMP,
    first_seen_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BUSINESS_LEDGER_TABLE} (
    business_key STRING,
    first_source_file_name STRING,
    first_source_line_no BIGINT,
    first_source_line_identity STRING,
    first_outcome_reason STRING,
    first_seen_event_ts TIMESTAMP,
    first_seen_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
    stream_name STRING,
    max_event_ts TIMESTAMP,
    last_batch_id BIGINT,
    updated_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
    batch_id BIGINT,
    audit_ts TIMESTAMP,
    source_file_name STRING,
    input_rows BIGINT,
    trusted_rows BIGINT,
    quarantined_rows BIGINT,
    quality_summary STRING,
    max_event_ts_seen TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
MERGE INTO {CONTROL_TABLE} AS t
USING (
    SELECT
        'cartflow_order_status_v1' AS stream_name,
        CAST(NULL AS TIMESTAMP) AS max_event_ts,
        CAST(NULL AS BIGINT) AS last_batch_id,
        current_timestamp() AS updated_at
) AS s
ON t.stream_name = s.stream_name
WHEN NOT MATCHED THEN INSERT *
""")

print("Week-10 Delta tables are ready.")

## 7. Auto Loader → Bronze

Auto Loader reads each JSONL physical line as text. The explicit event schema is applied after ingestion. No business deduplication is performed in Bronze.

In [ ]:
def write_bronze_batch(raw_batch, batch_id):
    if raw_batch.limit(1).count() == 0:
        return

    batch = (
        raw_batch
        .select(
            F.col("value").alias("raw_json"),
            F.col("_metadata.file_path").alias("source_file_path"),
            F.col("_metadata.file_name").alias("source_file_name"),
            F.current_timestamp().alias("ingested_at"),
        )
        .join(
            F.broadcast(
                line_manifest.select(
                    "source_file_name",
                    "raw_json",
                    "source_line_no",
                    "source_line_identity",
                )
            ),
            ["source_file_name", "raw_json"],
            "left",
        )
    )

    missing_line_identity = (
        batch.where(F.col("source_line_identity").isNull()).limit(1).count()
    )
    if missing_line_identity:
        raise RuntimeError(
            f"Bronze batch {batch_id} contains a raw line that is missing from the "
            "physical-line manifest."
        )

    allowed_keys_array = F.array(*[F.lit(x) for x in ALLOWED_EVENT_FIELDS])
    required_keys_array = F.array(*[F.lit(x) for x in REQUIRED_JSON_KEYS])

    parsed = (
        batch
        .withColumn("_event", F.from_json("raw_json", event_schema, {"mode": "PERMISSIVE"}))
        .withColumn("_raw_map", F.from_json("raw_json", raw_map_schema, {"mode": "PERMISSIVE"}))
        .withColumn(
            "_missing_keys",
            F.array_except(required_keys_array, F.map_keys(F.col("_raw_map")))
        )
        .withColumn(
            "_unexpected_keys",
            F.array_except(F.map_keys(F.col("_raw_map")), allowed_keys_array)
        )
    )

    bronze_df = (
        parsed
        .select(
            "source_file_path",
            "source_file_name",
            "source_line_no",
            "source_line_identity",
            "raw_json",
            "ingested_at",
            "_event.*",
            F.col("_missing_keys"),
            F.col("_unexpected_keys"),
            F.col("_event").alias("_event_struct"),
        )
        .withColumn(
            "schema_error_flag",
            (
                F.col("_event_struct").isNull()
                | F.col("event_id").isNull()
                | F.col("schema_version").isNull()
                | F.col("event_ts").isNull()
                | F.col("event_type").isNull()
                | F.col("order_id").isNull()
                | F.col("seller_id").isNull()
                | F.col("order_item_id").isNull()
                | F.col("previous_status").isNull()
                | F.col("event_sequence_no").isNull()
                | (F.size(F.coalesce(F.col("_missing_keys"), F.array())) > 0)
                | (F.size(F.coalesce(F.col("_unexpected_keys"), F.array())) > 0)
            )
        )
        .withColumn(
            "schema_error_detail",
            F.concat_ws(
                "; ",
                F.when(F.col("_event_struct").isNull(), F.lit("malformed_json")),
                F.when(
                    F.size(F.coalesce(F.col("_missing_keys"), F.array())) > 0,
                    F.concat(
                        F.lit("missing_keys="),
                        F.concat_ws(",", F.array_sort(F.col("_missing_keys")))
                    )
                ),
                F.when(
                    F.size(F.coalesce(F.col("_unexpected_keys"), F.array())) > 0,
                    F.concat(
                        F.lit("unexpected_keys="),
                        F.concat_ws(",", F.array_sort(F.col("_unexpected_keys")))
                    )
                ),
                F.when(
                    F.col("schema_version").isNull(),
                    F.lit("schema_version_null_or_invalid")
                ),
                F.when(
                    F.col("event_ts").isNull(),
                    F.lit("event_ts_null_or_invalid")
                ),
                F.when(
                    F.col("event_sequence_no").isNull(),
                    F.lit("event_sequence_no_null_or_invalid")
                ),
            )
        )
        .withColumn("bronze_batch_id", F.lit(int(batch_id)))
        .drop("_missing_keys", "_unexpected_keys", "_event_struct")
    )

    bronze_df.createOrReplaceTempView("_week10_bronze_batch")

    spark.sql(f"""
    MERGE INTO {BRONZE_TABLE} AS t
    USING _week10_bronze_batch AS s
    ON t.source_line_identity = s.source_line_identity
    WHEN NOT MATCHED THEN INSERT *
    """)

bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.includeExistingFiles", "true")
    .option("cloudFiles.maxFilesPerTrigger", MAX_FILES_PER_TRIGGER)
    .schema(raw_line_schema)
    .load(INPUT_DIR)
)

bronze_query = (
    bronze_stream.writeStream
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CHECKPOINT_DIR)
    .foreachBatch(write_bronze_batch)
    .start()
)

bronze_query.processAllAvailable()
bronze_query.stop()

print("Drop-01 Bronze ingestion completed using the stable checkpoint.")

## 8. Drop-01 Bronze evidence

In [ ]:
display(
    spark.sql(f"""
    SELECT
        source_file_name,
        COUNT(*) AS bronze_rows,
        COUNT(DISTINCT source_line_identity) AS distinct_line_identities,
        MIN(source_line_no) AS first_line,
        MAX(source_line_no) AS last_line
    FROM {BRONZE_TABLE}
    GROUP BY source_file_name
    ORDER BY source_file_name
    """)
)

display(
    spark.sql(f"""
    SELECT
        source_file_name,
        source_line_no,
        event_id,
        schema_error_flag,
        schema_error_detail
    FROM {BRONZE_TABLE}
    WHERE schema_error_flag = true
    ORDER BY source_file_name, source_line_no
    """)
)

## 9. Quality-routing function

Quality precedence is deterministic:

1. schema error
2. future event time
3. late event
4. duplicate event ID
5. duplicate order/status/sequence
6. orphaned reference
7. invalid event type
8. invalid transition
9. trusted

In [ ]:
def route_quality_batch(batch_df, batch_id):
    if batch_df.limit(1).count() == 0:
        return

    previous_max_ts = (
        spark.table(CONTROL_TABLE)
        .where(F.col("stream_name") == "cartflow_order_status_v1")
        .select("max_event_ts")
        .first()["max_event_ts"]
    )

    df = batch_df.withColumn(
        "_business_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("order_id"), F.lit("")),
                F.coalesce(F.col("previous_status"), F.lit("")),
                F.coalesce(F.col("new_status"), F.lit("")),
                F.coalesce(F.col("event_sequence_no").cast("string"), F.lit("")),
            ),
            256,
        ),
    )

    event_window = Window.partitionBy("event_id").orderBy(F.col("source_line_no"))
    business_window = Window.partitionBy("_business_key").orderBy(F.col("source_line_no"))

    df = (
        df
        .withColumn("_event_rank", F.row_number().over(event_window))
        .withColumn("_business_rank", F.row_number().over(business_window))
    )

    seen_events = (
        spark.table(EVENT_LEDGER_TABLE)
        .select(F.col("event_id").alias("_seen_event_id"))
        .where(F.col("event_id").isNotNull())
        .dropDuplicates()
    )

    seen_business = (
        spark.table(BUSINESS_LEDGER_TABLE)
        .select(F.col("business_key").alias("_seen_business_key"))
        .where(F.col("business_key").isNotNull())
        .dropDuplicates()
    )

    df = (
        df
        .join(
            seen_events,
            F.col("event_id") == F.col("_seen_event_id"),
            "left",
        )
        .join(
            seen_business,
            F.col("_business_key") == F.col("_seen_business_key"),
            "left",
        )
        .join(
            orders_ref.withColumn("_order_exists", F.lit(1)),
            "order_id",
            "left",
        )
        .join(
            items_ref.withColumn("_item_exists", F.lit(1)),
            "order_item_id",
            "left",
        )
        .join(
            sellers_ref.withColumn("_seller_exists", F.lit(1)),
            "seller_id",
            "left",
        )
    )

    df = df.withColumn(
        "_future_event",
        F.col("event_ts").isNotNull() & (F.col("event_ts") > F.current_timestamp()),
    )

    df = df.withColumn(
        "_watermark_candidate_ts",
        F.when(
            (~F.col("schema_error_flag")) & (~F.col("_future_event")),
            F.col("event_ts"),
        ),
    )

    prior_rows = (
        Window
        .partitionBy("source_file_name")
        .orderBy(F.col("source_line_no"))
        .rowsBetween(Window.unboundedPreceding, -1)
    )

    df = df.withColumn(
        "_prior_file_max_event_ts",
        F.max("_watermark_candidate_ts").over(prior_rows),
    )

    seed_ts = (
        F.lit(previous_max_ts)
        if previous_max_ts is not None
        else F.lit(None).cast("timestamp")
    )

    df = df.withColumn(
        "_late_reference_ts",
        F.greatest(seed_ts, F.col("_prior_file_max_event_ts")),
    )

    df = df.withColumn(
        "_late_event",
        F.col("event_ts").isNotNull()
        & F.col("_late_reference_ts").isNotNull()
        & (
            F.col("event_ts")
            < F.col("_late_reference_ts") - F.expr("INTERVAL 30 MINUTES")
        ),
    )

    valid_transition = (
        (
            (F.col("previous_status") == "approved")
            & (F.col("new_status") == "shipped")
        )
        |
        (
            (F.col("previous_status") == "shipped")
            & (F.col("new_status") == "delivered")
        )
    )

    df = df.withColumn(
        "_quality_reason",
        F.when(F.col("schema_error_flag"), F.lit("malformed_or_schema_error"))
        .when(F.col("_future_event"), F.lit("future_event_time"))
        .when(F.col("_late_event"), F.lit("late_event"))
        .when(
            (F.col("event_id").isNotNull() & (F.col("_event_rank") > 1))
            | F.col("_seen_event_id").isNotNull(),
            F.lit("duplicate_event_id"),
        )
        .when(
            (F.col("_business_rank") > 1)
            | F.col("_seen_business_key").isNotNull(),
            F.lit("duplicate_order_status_sequence"),
        )
        .when(
            F.col("_order_exists").isNull()
            | F.col("_item_exists").isNull()
            | F.col("_seller_exists").isNull(),
            F.lit("orphaned_reference"),
        )
        .when(
            F.col("event_type") != "order_status_changed",
            F.lit("invalid_event_type"),
        )
        .when(~valid_transition, F.lit("invalid_transition"))
        .otherwise(F.lit("trusted")),
    )

    df = df.withColumn(
        "_quality_detail",
        F.when(
            F.col("_quality_reason") == "late_event",
            F.concat(
                F.lit("event_ts="),
                F.col("event_ts").cast("string"),
                F.lit("; late_reference="),
                F.col("_late_reference_ts").cast("string"),
                F.lit("; watermark_delay=30 minutes"),
            ),
        )
        .when(
            F.col("_quality_reason") == "duplicate_event_id",
            F.lit("event_id already observed or repeated in the current drop"),
        )
        .when(
            F.col("_quality_reason") == "duplicate_order_status_sequence",
            F.lit("order/status/sequence identity already observed or repeated"),
        )
        .when(
            F.col("_quality_reason") == "orphaned_reference",
            F.concat(
                F.lit("order_exists="),
                F.coalesce(F.col("_order_exists").cast("string"), F.lit("false")),
                F.lit("; item_exists="),
                F.coalesce(F.col("_item_exists").cast("string"), F.lit("false")),
                F.lit("; seller_exists="),
                F.coalesce(F.col("_seller_exists").cast("string"), F.lit("false")),
            ),
        )
        .when(
            F.col("_quality_reason") == "invalid_transition",
            F.concat(
                F.lit("previous_status="),
                F.coalesce(F.col("previous_status"), F.lit("<null>")),
                F.lit("; new_status="),
                F.coalesce(F.col("new_status"), F.lit("<null>")),
            ),
        )
        .when(
            F.col("_quality_reason") == "future_event_time",
            F.lit("event_ts is later than the current processing time"),
        )
        .otherwise(F.col("schema_error_detail")),
    )

    output_cols = [
        "event_id",
        "schema_version",
        "event_ts",
        "event_type",
        "order_id",
        "seller_id",
        "order_item_id",
        "previous_status",
        "new_status",
        "status_reason",
        "promised_delivery_ts",
        "location_code",
        "event_sequence_no",
        "producer_run_id",
        "raw_json",
        "source_file_path",
        "source_file_name",
        "source_line_no",
        "source_line_identity",
    ]

    trusted_df = (
        df.where(F.col("_quality_reason") == "trusted")
        .select(
            *output_cols,
            F.lit("trusted").alias("quality_reason"),
            F.current_timestamp().alias("trusted_at"),
            F.lit(int(batch_id)).alias("quality_batch_id"),
        )
    )

    quarantine_df = (
        df.where(F.col("_quality_reason") != "trusted")
        .select(
            *output_cols,
            F.col("_quality_reason").alias("quality_reason"),
            F.col("_quality_detail").alias("quality_detail"),
            F.current_timestamp().alias("quarantined_at"),
            F.lit(int(batch_id)).alias("quality_batch_id"),
        )
    )

    trusted_df.createOrReplaceTempView("_week10_trusted_batch")
    quarantine_df.createOrReplaceTempView("_week10_quarantine_batch")

    spark.sql(f"""
    MERGE INTO {TRUSTED_TABLE} AS t
    USING _week10_trusted_batch AS s
    ON t.source_line_identity = s.source_line_identity
    WHEN NOT MATCHED THEN INSERT *
    """)

    spark.sql(f"""
    MERGE INTO {QUARANTINE_TABLE} AS t
    USING _week10_quarantine_batch AS s
    ON t.source_line_identity = s.source_line_identity
    WHEN NOT MATCHED THEN INSERT *
    """)

    event_ledger_rows = (
        df
        .where(F.col("event_id").isNotNull() & (F.col("_event_rank") == 1))
        .select(
            "event_id",
            F.col("source_file_name").alias("first_source_file_name"),
            F.col("source_line_no").alias("first_source_line_no"),
            F.col("source_line_identity").alias("first_source_line_identity"),
            F.col("_quality_reason").alias("first_outcome_reason"),
            F.col("event_ts").alias("first_seen_event_ts"),
            F.current_timestamp().alias("first_seen_at"),
        )
        .dropDuplicates(["event_id"])
    )
    event_ledger_rows.createOrReplaceTempView("_week10_event_ledger_batch")

    spark.sql(f"""
    MERGE INTO {EVENT_LEDGER_TABLE} AS t
    USING _week10_event_ledger_batch AS s
    ON t.event_id = s.event_id
    WHEN NOT MATCHED THEN INSERT *
    """)

    business_ledger_rows = (
        df
        .where(F.col("_business_key").isNotNull() & (F.col("_business_rank") == 1))
        .select(
            F.col("_business_key").alias("business_key"),
            F.col("source_file_name").alias("first_source_file_name"),
            F.col("source_line_no").alias("first_source_line_no"),
            F.col("source_line_identity").alias("first_source_line_identity"),
            F.col("_quality_reason").alias("first_outcome_reason"),
            F.col("event_ts").alias("first_seen_event_ts"),
            F.current_timestamp().alias("first_seen_at"),
        )
        .dropDuplicates(["business_key"])
    )
    business_ledger_rows.createOrReplaceTempView("_week10_business_ledger_batch")

    spark.sql(f"""
    MERGE INTO {BUSINESS_LEDGER_TABLE} AS t
    USING _week10_business_ledger_batch AS s
    ON t.business_key = s.business_key
    WHEN NOT MATCHED THEN INSERT *
    """)

    batch_watermark_max = (
        df
        .where((~F.col("schema_error_flag")) & (~F.col("_future_event")))
        .agg(F.max("event_ts").alias("batch_watermark_max"))
        .first()["batch_watermark_max"]
    )

    if batch_watermark_max is not None:
        new_max_ts = (
            batch_watermark_max
            if previous_max_ts is None
            else max(previous_max_ts, batch_watermark_max)
        )

        spark.sql(f"""
        UPDATE {CONTROL_TABLE}
        SET
            max_event_ts = TIMESTAMP('{new_max_ts.strftime("%Y-%m-%d %H:%M:%S")}'),
            last_batch_id = {int(batch_id)},
            updated_at = current_timestamp()
        WHERE stream_name = 'cartflow_order_status_v1'
        """)

    source_files = (
        df.select("source_file_name")
        .where(F.col("source_file_name").isNotNull())
        .distinct()
        .collect()
    )

    if len(source_files) != 1:
        raise RuntimeError(
            "Expected exactly one source file per controlled microbatch; "
            f"observed {[r['source_file_name'] for r in source_files]}"
        )

    source_file = source_files[0]["source_file_name"]

    quality_summary = (
        df.groupBy("_quality_reason")
        .count()
        .orderBy("_quality_reason")
        .collect()
    )
    quality_text = "; ".join(
        f"{r['_quality_reason']}={r['count']}" for r in quality_summary
    )

    audit_row = spark.createDataFrame(
        [(
            int(batch_id),
            datetime.now(timezone.utc).replace(tzinfo=None),
            source_file,
            int(df.count()),
            int(trusted_df.count()),
            int(quarantine_df.count()),
            quality_text,
            batch_watermark_max,
        )],
        [
            "batch_id",
            "audit_ts",
            "source_file_name",
            "input_rows",
            "trusted_rows",
            "quarantined_rows",
            "quality_summary",
            "max_event_ts_seen",
        ],
    )

    audit_row.createOrReplaceTempView("_week10_audit_batch")

    spark.sql(f"""
    MERGE INTO {AUDIT_TABLE} AS t
    USING _week10_audit_batch AS s
    ON t.batch_id = s.batch_id
    WHEN NOT MATCHED THEN INSERT *
    """)

## 10. Start the quality-routing stream for Drop 01

The downstream streaming DataFrame explicitly carries the required 30-minute event-time watermark.

In [ ]:
quality_stream = (
    spark.readStream
    .table(BRONZE_TABLE)
    .withWatermark("event_ts", WATERMARK_DELAY)
)

quality_query = (
    quality_stream.writeStream
    .outputMode("append")
    .option("checkpointLocation", QUALITY_CHECKPOINT_DIR)
    .foreachBatch(route_quality_batch)
    .start()
)

quality_query.processAllAvailable()
quality_query.stop()

print("Drop-01 quality routing completed.")

## 11. Drop-01 quality and lineage evidence

In [ ]:
print("Drop-01 quality summary:")
display(
    spark.sql(f"""
    SELECT
        quality_reason,
        COUNT(*) AS rows
    FROM {QUARANTINE_TABLE}
    WHERE source_file_name = 'order_status_drop_01.json'
    GROUP BY quality_reason
    ORDER BY quality_reason
    """)
)

print("Drop-01 trusted output:")
display(
    spark.sql(f"""
    SELECT
        source_file_name,
        COUNT(*) AS trusted_rows
    FROM {TRUSTED_TABLE}
    WHERE source_file_name = 'order_status_drop_01.json'
    GROUP BY source_file_name
    """)
)

print("Drop-01 audit:")
display(
    spark.sql(f"""
    SELECT *
    FROM {AUDIT_TABLE}
    WHERE source_file_name = 'order_status_drop_01.json'
    ORDER BY batch_id
    """)
)

print("Drop-01 quarantined rows with physical line identity:")
display(
    spark.sql(f"""
    SELECT
        source_file_name,
        source_line_no,
        source_line_identity,
        event_id,
        quality_reason,
        quality_detail
    FROM {QUARANTINE_TABLE}
    WHERE source_file_name = 'order_status_drop_01.json'
    ORDER BY source_line_no
    """)
)

## 12. Drop-01 deduplication evidence

In [ ]:
print("Repeated event_id values retained in Bronze:")
display(
    spark.sql(f"""
    SELECT
        event_id,
        COUNT(*) AS occurrences,
        COLLECT_LIST(
            NAMED_STRUCT(
                'source_line_no', source_line_no,
                'source_line_identity', source_line_identity
            )
        ) AS physical_lines
    FROM {BRONZE_TABLE}
    WHERE source_file_name = 'order_status_drop_01.json'
      AND event_id IS NOT NULL
    GROUP BY event_id
    HAVING COUNT(*) > 1
    ORDER BY event_id
    """)
)

print("Repeated order/status/sequence identities retained in Bronze:")
display(
    spark.sql(f"""
    SELECT
        order_id,
        previous_status,
        new_status,
        event_sequence_no,
        COUNT(*) AS occurrences
    FROM {BRONZE_TABLE}
    WHERE source_file_name = 'order_status_drop_01.json'
    GROUP BY
        order_id,
        previous_status,
        new_status,
        event_sequence_no
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC, order_id
    """)
)

## 13. Stop state is retained before Drop 02

The same Bronze and quality checkpoints will be reused after Drop 02 is released.

In [ ]:
print("Bronze checkpoint retained:", BRONZE_CHECKPOINT_DIR)
print("Quality checkpoint retained:", QUALITY_CHECKPOINT_DIR)
display(spark.table(CONTROL_TABLE))

## 14. Release Drop 02 into the same landing directory

Drop 01 is not moved or rewritten. Only the held Drop 02 file is released.

In [ ]:
hold_paths = {x.path for x in dbutils.fs.ls(HOLD_DIR)}

if DROP02_HOLD_PATH not in hold_paths:
    raise FileNotFoundError(
        "Drop 02 is not available in the hold directory: " + DROP02_HOLD_PATH
    )

dbutils.fs.mv(DROP02_HOLD_PATH, DROP02_PATH)

print("Drop 02 released into the same landing directory.")
display(dbutils.fs.ls(INPUT_DIR))

## 15. Restart Auto Loader with the SAME Bronze checkpoint

In [ ]:
bronze_stream_drop2 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.includeExistingFiles", "true")
    .option("cloudFiles.maxFilesPerTrigger", MAX_FILES_PER_TRIGGER)
    .schema(raw_line_schema)
    .load(INPUT_DIR)
)

bronze_query_drop2 = (
    bronze_stream_drop2.writeStream
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CHECKPOINT_DIR)
    .foreachBatch(write_bronze_batch)
    .start()
)

bronze_query_drop2.processAllAvailable()
bronze_query_drop2.stop()

print("Drop-02 Bronze ingestion completed with the SAME stable checkpoint.")

## 16. Drop-02 Bronze evidence

In [ ]:
display(
    spark.sql(f"""
    SELECT
        source_file_name,
        COUNT(*) AS bronze_rows,
        COUNT(DISTINCT source_line_identity) AS distinct_line_identities,
        MIN(source_line_no) AS first_line,
        MAX(source_line_no) AS last_line
    FROM {BRONZE_TABLE}
    GROUP BY source_file_name
    ORDER BY source_file_name
    """)
)

display(
    spark.sql(f"""
    SELECT
        source_file_name,
        source_line_no,
        event_id,
        schema_error_flag,
        schema_error_detail
    FROM {BRONZE_TABLE}
    WHERE source_file_name = 'order_status_drop_02.json'
      AND schema_error_flag = true
    ORDER BY source_line_no
    """)
)

## 17. Process Drop 02 through the quality-routing stream using the SAME quality checkpoint

In [ ]:
quality_stream_drop2 = (
    spark.readStream
    .table(BRONZE_TABLE)
    .withWatermark("event_ts", WATERMARK_DELAY)
)

quality_query_drop2 = (
    quality_stream_drop2.writeStream
    .outputMode("append")
    .option("checkpointLocation", QUALITY_CHECKPOINT_DIR)
    .foreachBatch(route_quality_batch)
    .start()
)

quality_query_drop2.processAllAvailable()
quality_query_drop2.stop()

print("Drop-02 quality routing completed with the existing quality checkpoint.")

## 18. Cumulative Drop-02 quality and lineage evidence

In [ ]:
print("Drop-02 quality summary:")
display(
    spark.sql(f"""
    SELECT
        quality_reason,
        COUNT(*) AS rows
    FROM {QUARANTINE_TABLE}
    WHERE source_file_name = 'order_status_drop_02.json'
    GROUP BY quality_reason
    ORDER BY quality_reason
    """)
)

print("Drop-02 trusted output:")
display(
    spark.sql(f"""
    SELECT
        source_file_name,
        COUNT(*) AS trusted_rows
    FROM {TRUSTED_TABLE}
    WHERE source_file_name = 'order_status_drop_02.json'
    GROUP BY source_file_name
    """)
)

print("Drop-02 audit:")
display(
    spark.sql(f"""
    SELECT *
    FROM {AUDIT_TABLE}
    WHERE source_file_name = 'order_status_drop_02.json'
    ORDER BY batch_id
    """)
)

print("Drop-02 quarantined rows with physical line identity:")
display(
    spark.sql(f"""
    SELECT
        source_file_name,
        source_line_no,
        source_line_identity,
        event_id,
        quality_reason,
        quality_detail
    FROM {QUARANTINE_TABLE}
    WHERE source_file_name = 'order_status_drop_02.json'
    ORDER BY source_line_no
    """)
)

## 19. Cross-drop deduplication evidence

In [ ]:
print("Event IDs observed more than once across the two drops:")
display(
    spark.sql(f"""
    SELECT
        event_id,
        COUNT(*) AS bronze_occurrences,
        COLLECT_LIST(
            NAMED_STRUCT(
                'source_file_name', source_file_name,
                'source_line_no', source_line_no,
                'source_line_identity', source_line_identity
            )
        ) AS physical_lines
    FROM {BRONZE_TABLE}
    WHERE event_id IS NOT NULL
    GROUP BY event_id
    HAVING COUNT(*) > 1
    ORDER BY event_id
    """)
)

print("Event-ID dedup ledger:")
display(
    spark.sql(f"""
    SELECT *
    FROM {EVENT_LEDGER_TABLE}
    ORDER BY event_id
    LIMIT 20
    """)
)

print("Order/status/sequence dedup ledger:")
display(
    spark.sql(f"""
    SELECT *
    FROM {BUSINESS_LEDGER_TABLE}
    ORDER BY business_key
    LIMIT 20
    """)
)

## 20. No-new-file recovery proof — BEFORE restart

Both streaming queries are stopped. Capture all governed output counts before restarting them with the exact same checkpoints.

In [ ]:
bronze_before = spark.table(BRONZE_TABLE).count()
trusted_before = spark.table(TRUSTED_TABLE).count()
quarantine_before = spark.table(QUARANTINE_TABLE).count()
audit_before = spark.table(AUDIT_TABLE).count()
event_ledger_before = spark.table(EVENT_LEDGER_TABLE).count()
business_ledger_before = spark.table(BUSINESS_LEDGER_TABLE).count()

before_counts = spark.createDataFrame(
    [(
        bronze_before,
        trusted_before,
        quarantine_before,
        audit_before,
        event_ledger_before,
        business_ledger_before,
    )],
    [
        "bronze_before",
        "trusted_before",
        "quarantine_before",
        "audit_before",
        "event_ledger_before",
        "business_ledger_before",
    ],
)

display(before_counts)

## 21. No-new-file recovery proof — restart Bronze with the SAME checkpoint

In [ ]:
bronze_recovery_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.includeExistingFiles", "true")
    .option("cloudFiles.maxFilesPerTrigger", MAX_FILES_PER_TRIGGER)
    .schema(raw_line_schema)
    .load(INPUT_DIR)
)

bronze_recovery_query = (
    bronze_recovery_stream.writeStream
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CHECKPOINT_DIR)
    .foreachBatch(write_bronze_batch)
    .start()
)

bronze_recovery_query.processAllAvailable()
bronze_recovery_query.stop()

print("Bronze no-new-file recovery run completed.")

## 22. No-new-file recovery proof — restart quality routing with the SAME checkpoint

In [ ]:
quality_recovery_stream = (
    spark.readStream
    .table(BRONZE_TABLE)
    .withWatermark("event_ts", WATERMARK_DELAY)
)

quality_recovery_query = (
    quality_recovery_stream.writeStream
    .outputMode("append")
    .option("checkpointLocation", QUALITY_CHECKPOINT_DIR)
    .foreachBatch(route_quality_batch)
    .start()
)

quality_recovery_query.processAllAvailable()
quality_recovery_query.stop()

print("Quality no-new-file recovery run completed.")

## 23. No-new-file recovery proof — AFTER restart

In [ ]:
bronze_after = spark.table(BRONZE_TABLE).count()
trusted_after = spark.table(TRUSTED_TABLE).count()
quarantine_after = spark.table(QUARANTINE_TABLE).count()
audit_after = spark.table(AUDIT_TABLE).count()
event_ledger_after = spark.table(EVENT_LEDGER_TABLE).count()
business_ledger_after = spark.table(BUSINESS_LEDGER_TABLE).count()

recovery_results = spark.createDataFrame(
    [
        ("Bronze rows unchanged", bronze_before, bronze_after, bronze_before == bronze_after),
        ("Trusted rows unchanged", trusted_before, trusted_after, trusted_before == trusted_after),
        ("Quarantine rows unchanged", quarantine_before, quarantine_after, quarantine_before == quarantine_after),
        ("Audit rows unchanged", audit_before, audit_after, audit_before == audit_after),
        ("Event ledger unchanged", event_ledger_before, event_ledger_after, event_ledger_before == event_ledger_after),
        ("Business ledger unchanged", business_ledger_before, business_ledger_after, business_ledger_before == business_ledger_after),
    ],
    ["check", "before_count", "after_count", "observed_stable"],
)

display(recovery_results)

if recovery_results.where(F.col("observed_stable") == False).limit(1).count() > 0:
    raise RuntimeError("No-new-file recovery proof failed: at least one governed count changed.")

print("No-new-file recovery counts are stable.")

## 24. Final governed-output reconciliation

Every Bronze physical line must have exactly one downstream outcome: Trusted or Quarantine.

In [ ]:
bronze_count = spark.table(BRONZE_TABLE).count()
trusted_count = spark.table(TRUSTED_TABLE).count()
quarantine_count = spark.table(QUARANTINE_TABLE).count()

duplicate_bronze_line_ids = (
    spark.table(BRONZE_TABLE)
    .groupBy("source_line_identity")
    .count()
    .where("count > 1")
    .count()
)

duplicate_trusted_line_ids = (
    spark.table(TRUSTED_TABLE)
    .groupBy("source_line_identity")
    .count()
    .where("count > 1")
    .count()
)

duplicate_quarantine_line_ids = (
    spark.table(QUARANTINE_TABLE)
    .groupBy("source_line_identity")
    .count()
    .where("count > 1")
    .count()
)

trusted_plus_quarantine = trusted_count + quarantine_count

summary = spark.createDataFrame(
    [
        ("Bronze physical lines", bronze_count),
        ("Trusted outcomes", trusted_count),
        ("Quarantine outcomes", quarantine_count),
        ("Trusted + Quarantine", trusted_plus_quarantine),
        ("Duplicate Bronze line identities", duplicate_bronze_line_ids),
        ("Duplicate Trusted line identities", duplicate_trusted_line_ids),
        ("Duplicate Quarantine line identities", duplicate_quarantine_line_ids),
    ],
    ["metric", "value"],
)

display(summary)

if trusted_plus_quarantine != bronze_count:
    raise RuntimeError(
        "Downstream outcome reconciliation failed: Trusted + Quarantine != Bronze."
    )

if duplicate_bronze_line_ids != 0 or duplicate_trusted_line_ids != 0 or duplicate_quarantine_line_ids != 0:
    raise RuntimeError("A physical source-line identity is duplicated in a governed output.")

print("Governed outcome reconciliation passed.")

## 25. Final source-file, quality, lineage and watermark evidence

In [ ]:
print("Final source-file counts:")
display(
    spark.sql(f"""
    SELECT
        source_file_name,
        COUNT(*) AS bronze_rows,
        COUNT(DISTINCT source_line_identity) AS distinct_line_identities,
        MIN(source_line_no) AS first_line,
        MAX(source_line_no) AS last_line
    FROM {BRONZE_TABLE}
    GROUP BY source_file_name
    ORDER BY source_file_name
    """)
)

print("Final quality routing:")
display(
    spark.sql(f"""
    SELECT
        source_file_name,
        quality_reason,
        COUNT(*) AS rows
    FROM {QUARANTINE_TABLE}
    GROUP BY source_file_name, quality_reason
    ORDER BY source_file_name, quality_reason
    """)
)

print("Final trusted output:")
display(
    spark.sql(f"""
    SELECT
        source_file_name,
        COUNT(*) AS trusted_rows
    FROM {TRUSTED_TABLE}
    GROUP BY source_file_name
    ORDER BY source_file_name
    """)
)

print("Final watermark control state:")
display(spark.table(CONTROL_TABLE))

print("Final run audit:")
display(
    spark.sql(f"""
    SELECT *
    FROM {AUDIT_TABLE}
    ORDER BY batch_id
    """)
)

## 26. Final validation checklist

These checks produce the evidence; no expected quality counts are hard-coded.

In [ ]:
validation_rows = [
    (
        "Drop 01 physical lines = 100",
        spark.sql(f"""
            SELECT COUNT(*) AS n
            FROM {BRONZE_TABLE}
            WHERE source_file_name = 'order_status_drop_01.json'
        """).first()["n"] == 100,
    ),
    (
        "Drop 02 physical lines = 100",
        spark.sql(f"""
            SELECT COUNT(*) AS n
            FROM {BRONZE_TABLE}
            WHERE source_file_name = 'order_status_drop_02.json'
        """).first()["n"] == 100,
    ),
    ("Trusted + Quarantine = Bronze", trusted_plus_quarantine == bronze_count),
    ("Bronze line identities unique", duplicate_bronze_line_ids == 0),
    ("Trusted line identities unique", duplicate_trusted_line_ids == 0),
    ("Quarantine line identities unique", duplicate_quarantine_line_ids == 0),
    ("No-new-file Bronze count stable", bronze_before == bronze_after),
    ("No-new-file Trusted count stable", trusted_before == trusted_after),
    ("No-new-file Quarantine count stable", quarantine_before == quarantine_after),
    ("No-new-file Audit count stable", audit_before == audit_after),
    ("Event ledger stable on no-new-file rerun", event_ledger_before == event_ledger_after),
    ("Business ledger stable on no-new-file rerun", business_ledger_before == business_ledger_after),
]

validation_df = spark.createDataFrame(
    [(name, bool(result)) for name, result in validation_rows],
    ["validation_check", "passed"],
)

display(validation_df)

failed = validation_df.where(F.col("passed") == False).count()

if failed != 0:
    raise RuntimeError(f"{failed} final Week-10 validation check(s) failed.")

print("All final Week-10 validation checks passed.")

## 27. Week-10 evidence checklist

Capture genuine Databricks screenshots from the executed notebook for:

1. Drop-01 source-file/physical-line counts.
2. Drop-01 quality routing and quarantined line identities.
3. Drop-02 source-file/physical-line counts.
4. Drop-02 quality routing and quarantined line identities.
5. Cross-drop event-ID dedup evidence.
6. Final Trusted + Quarantine reconciliation.
7. No-new-file recovery proof showing before/after counts unchanged.
8. Final watermark control state and run audit.

The notebook does not fabricate counts or screenshots. The values above become evidence only after the cells are actually executed in Databricks.

## 28. Week-10 completion boundary

When the final validation cell passes, this notebook provides the Week-10 evidence for:

- Drop 01 followed by Drop 02 incremental processing.
- Explicit event schema.
- Stable Auto Loader checkpoint:
  `stable/checkpoints/cartflow_order_status_v1`
- 30-minute event-time watermark.
- Event-ID deduplication.
- Order/status/sequence deduplication.
- Malformed/schema-error routing.
- Late-event routing.
- Duplicate routing.
- Orphaned-reference routing.
- Invalid-event-type and invalid-transition routing.
- Physical filename + physical line number + stable line identity.
- Trusted and quarantined persisted outcomes.
- Same-checkpoint no-new-file recovery proof.
- No changes to the Week-9 Gold/Power BI layer.

Do not modify Week-9 Power BI or Gold artifacts inside this notebook.